# Siniestros viales en la Ciudad de Buenos Aires

Análisis exploratorio de los siniestros viales con víctimas registrados en la Ciudad de Buenos Aires desde 2019, con los datos abiertos del Observatorio de Movilidad y Seguridad Vial del GCBA.

**Preguntas:**

1. ¿Cómo evolucionaron los siniestros y las víctimas año a año?
2. ¿Quiénes son las víctimas más vulnerables?
3. ¿En qué horarios y días se concentran los siniestros mortales?
4. ¿Dónde ocurren?

Los números del texto se calculan en cada ejecución, así el informe se actualiza solo cuando se publican datos nuevos.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from siniestros.datos import DIAS, cargar

plt.rcParams.update({
    "figure.figsize": (10, 4.5), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold", "axes.titlesize": 12,
})
AZUL, ROJO, GRIS = "#2563eb", "#dc2626", "#9ca3af"
fmt = lambda n: f"{n:,.0f}".replace(",", ".")
pct = lambda x: f"{x:.1f}%".replace(".", ",")

hechos, victimas = cargar()
print(f"Hechos: {len(hechos):,} · Víctimas: {len(victimas):,} · Período: {hechos['fecha'].min():%d/%m/%Y} a {hechos['fecha'].max():%d/%m/%Y}")

## 1. Calidad de los datos

Antes de analizar, cuánto falta en cada variable. Los valores `SD` (sin dato) se tratan como nulos.

In [ ]:
faltantes = pd.concat({
    "hechos": hechos[["victima", "contraparte", "gravedad", "comuna", "hh", "latitud"]].isna().mean(),
    "victimas": victimas[["victima", "sexo_victima", "edad_victima", "gravedad", "rol_victima"]].isna().mean(),
}).mul(100).round(1).rename("% faltante").to_frame()
faltantes

## 2. Evolución anual

In [ ]:
anual = victimas.groupby("anio").agg(victimas=("id_hecho", "size"), mortales=("mortal", "sum"))
anual["% mortales"] = (anual["mortales"] / anual["victimas"] * 100).round(1)

fig, ax = plt.subplots()
ax.bar(anual.index, anual["victimas"], color=AZUL, label="Víctimas totales")
ax2 = ax.twinx()
ax2.plot(anual.index, anual["mortales"], color=ROJO, marker="o", lw=2, label="Víctimas mortales")
ax2.spines["top"].set_visible(False); ax2.grid(False); ax2.set_ylim(bottom=0)
ax.set_title("Víctimas por año"); ax.set_ylabel("Víctimas"); ax2.set_ylabel("Mortales", color=ROJO)
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88), frameon=False)
plt.show()

completos = anual.loc[[a for a in anual.index if victimas.loc[victimas["anio"] == a, "fecha"].dt.month.max() == 12]]
if len(completos) >= 2:
    a0, a1 = completos.index[0], completos.index[-1]
    var = (completos.loc[a1, "mortales"] / completos.loc[a0, "mortales"] - 1) * 100
    display(Markdown(f"Entre {a0} y {a1} las víctimas mortales pasaron de **{fmt(completos.loc[a0, 'mortales'])}** a **{fmt(completos.loc[a1, 'mortales'])}** ({'+' if var > 0 else ''}{pct(var)})."))
anual

## 3. ¿Quiénes son las víctimas?

### Por medio de transporte

In [ ]:
por_medio = (victimas.dropna(subset=["victima"]).groupby("victima")
             .agg(victimas=("id_hecho", "size"), mortales=("mortal", "sum")))
por_medio["% del total de mortales"] = (por_medio["mortales"] / por_medio["mortales"].sum() * 100).round(1)
por_medio["letalidad %"] = (por_medio["mortales"] / por_medio["victimas"] * 100).round(2)
por_medio = por_medio.sort_values("mortales", ascending=False)

fig, ax = plt.subplots()
orden = por_medio.sort_values("mortales")
ax.barh(orden.index, orden["mortales"], color=[ROJO if i >= len(orden) - 2 else GRIS for i in range(len(orden))])
ax.set_title("Víctimas mortales por medio de transporte"); ax.set_xlabel("Víctimas mortales")
plt.show()

top = por_medio.head(2)
display(Markdown(
    f"**{top.index[0].title()}** y **{top.index[1].title()}** concentran el "
    f"**{pct(top['% del total de mortales'].sum())}** de las víctimas mortales. "
    f"La letalidad (mortales sobre víctimas) más alta es la de **{por_medio['letalidad %'].idxmax().title()}**: "
    f"{pct(por_medio['letalidad %'].max())} de sus víctimas muere."
))
por_medio

### Por edad y sexo (víctimas mortales)

In [ ]:
mortales = victimas[victimas["mortal"]]
bins = [0, 18, 30, 45, 60, 75, 120]
etiquetas = ["0-17", "18-29", "30-44", "45-59", "60-74", "75+"]
mortales = mortales.assign(rango=pd.cut(mortales["edad_victima"], bins, right=False, labels=etiquetas))
tabla = mortales.dropna(subset=["sexo_victima"]).pivot_table(index="rango", columns="sexo_victima", values="id_hecho",
                                                            aggfunc="size", fill_value=0, observed=False)

fig, ax = plt.subplots()
tabla.plot(kind="bar", ax=ax, color=[ROJO, AZUL, GRIS][: tabla.shape[1]], rot=0, width=0.8)
ax.set_title("Víctimas mortales por edad y sexo"); ax.set_xlabel("Edad"); ax.set_ylabel("Víctimas mortales")
ax.legend(title=None, frameon=False)
plt.show()

if len(mortales):
    sexo = mortales["sexo_victima"].value_counts(normalize=True).mul(100)
    display(Markdown(
        f"La edad mediana de las víctimas mortales es **{mortales['edad_victima'].median():.0f} años**. "
        f"El sexo más frecuente es **{sexo.index[0].lower()}** ({pct(sexo.iloc[0])})."
    ))

## 4. ¿Cuándo ocurren?

In [ ]:
graves = hechos[hechos["mortal"]]
calor = (graves.dropna(subset=["hh"]).assign(hh=lambda d: d["hh"].astype(int))
         .pivot_table(index="dia_semana", columns="hh", values="id_hecho", aggfunc="size", fill_value=0, observed=False)
         .reindex(index=DIAS, columns=range(24), fill_value=0))

fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(calor, aspect="auto", cmap="Reds")
ax.set_yticks(range(7), DIAS); ax.set_xticks(range(0, 24, 2)); ax.grid(False)
ax.set_xlabel("Hora del día"); ax.set_title("Siniestros mortales por día y hora")
fig.colorbar(im, ax=ax, label="Siniestros")
plt.show()

if len(graves):
    franja = graves["hh"].between(0, 6).mean() * 100
    finde = graves["dia_semana"].isin(["Sábado", "Domingo"]).mean() * 100
    display(Markdown(
        f"El **{pct(franja)}** de los siniestros mortales ocurre entre las 0 y las 6 h, "
        f"una franja con poco tránsito. El **{pct(finde)}** ocurre sábados o domingos (2 de 7 días = 28,6%)."
    ))

## 5. ¿Dónde ocurren?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [1.1, 1]})
geo = hechos.dropna(subset=["latitud", "longitud"])
ax1.scatter(geo["longitud"], geo["latitud"], s=2, color=GRIS, alpha=0.3, label="Con víctimas")
geo_m = geo[geo["mortal"]]
ax1.scatter(geo_m["longitud"], geo_m["latitud"], s=10, color=ROJO, label="Mortales")
ax1.set_aspect("equal"); ax1.set_title("Ubicación de los siniestros"); ax1.set_xticks([]); ax1.set_yticks([])
ax1.legend(frameon=False, loc="lower right")

por_calle = hechos[hechos["mortal"]].dropna(subset=["tipo_de_calle"])["tipo_de_calle"].value_counts()
ax2.barh(por_calle.index[::-1], por_calle.values[::-1], color=ROJO)
ax2.set_title("Siniestros mortales por tipo de vía")
plt.tight_layout(); plt.show()

por_comuna = (hechos.dropna(subset=["comuna"]).groupby(hechos["comuna"].astype("Int64"))
              .agg(siniestros=("id_hecho", "size"), mortales=("mortal", "sum"))
              .sort_values("mortales", ascending=False))
if len(por_calle):
    display(Markdown(
        f"El **{pct(por_calle.iloc[0] / por_calle.sum() * 100)}** de los siniestros mortales ocurre en **{por_calle.index[0].lower()}**. "
        f"La comuna con más siniestros mortales es la **{por_comuna.index[0]}**."
    ))
por_comuna.head(5)

## Conclusiones

In [ ]:
puntos = []
if len(por_medio) >= 2:
    puntos.append(f"**{top.index[0].title()}** y **{top.index[1].title()}** son los usuarios más expuestos: suman el {pct(top['% del total de mortales'].sum())} de las víctimas mortales.")
if len(graves):
    hora_pico = int(graves["hh"].mode().iloc[0])
    puntos.append(f"La hora con más siniestros mortales es la de las **{hora_pico} h**; entre las 0 y las 6 h ocurre el {pct(franja)} de los casos.")
if len(por_calle):
    puntos.append(f"El tipo de vía con más siniestros mortales es **{por_calle.index[0].lower()}**, con el {pct(por_calle.iloc[0] / por_calle.sum() * 100)} de los casos.")
display(Markdown("\n".join(f"- {p}" for p in puntos)))

## Notas metodológicas

- Fuente: [Buenos Aires Data, Siniestros viales](https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales) (CC BY 4.0). Ver las [notas para su uso](https://cdn.buenosaires.gob.ar/datosabiertos/datasets/transporte-y-obras-publicas/victimas-siniestros-viales/NOTAS_SINIESTROS_VIALES_2019-2025.pdf).
- Una víctima es *mortal* si fallece dentro de los 30 días del siniestro.
- La gravedad de un hecho es la de su víctima más grave.
- Los datos del último año pueden ser provisorios.